# BAE — Binary Autoencoder (broader-scope)

Reconstruction-only anomaly detector (no class head) with a binarized latent bottleneck (pushed toward 0/1 via a training penalty). Trained on the BROADER safe set (normal + motor_off + transition_on + transition_off) — moved here from the strict-only group since motor_off/transitions are legitimate states, not anomalies, and the decoder needs to have seen them.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn tensorflow joblib openpyxl matplotlib tf2onnx onnx

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input, Reshape
from tensorflow.keras.models import Model

MODEL_DISPLAY_NAME = 'binary Autoencoder'
MODEL_FILE_PREFIX = 'binary_autoencoder'
MODEL_OUTPUT_DIR = 'binary_autoencoder_output'

In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_PATH = r"../../data/Motor_data_model"
SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s",
    "PWM",
    "Irms",
    "Vrms",
    "Current_Variance",
    "Current_Crest_Factor",
    "Vib_RMS",
    "Vib_Min",
    "Vib_Q1",
    "Vib_Median",
    "Vib_Q3",
    "Vib_Max",
    "Vib_Variance",
    "Vib_Skewness",
]

SAFE_STATES = [
    "motor_off",
    "turning_off",
    "turning_on",
    "normal",
]

FAULT_STATES = [
    "electrical_fault",
    "mechanical_fault",
    "thermal_fault",
]

N_FEATURES = len(FEATURE_COLUMNS)
WINDOW_LEN = 30
WINDOW_STEP = 10
INPUT_SIZE = WINDOW_LEN * N_FEATURES
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
EPOCHS = 100
BATCH_SIZE = 64
RECON_THRESHOLD_PERCENTILE = 95
FILE_ANOMALY_FRACTION_THRESHOLD = 0.15
CLIP_SCALED_INPUT = False

OUTPUT_DIR = MODEL_OUTPUT_DIR
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Model:", MODEL_DISPLAY_NAME)
print("Safe states:", SAFE_STATES)
print("Fault states:", FAULT_STATES)

In [ ]:
# ==========================================================
# CELL 3 — map each file to one operating state
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def state_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if (
        "motor_being_off" in path_text
        or "turning_off" in path_text
        or "transition_off" in path_text
        or "off_spikes" in path_text
    ):
        return "turning_off"

    if (
        "motor_being_on" in path_text
        or "turning_on" in path_text
        or "transition_on" in path_text
        or "on_spikes" in path_text
    ):
        return "turning_on"

    if "/motor_off/" in path_text or "motor_off" in stem:
        return "motor_off"

    if (
        "electrical fault" in path_text
        or "electrical_fault" in path_text
        or "abnormal_current" in path_text
    ):
        return "electrical_fault"

    if (
        "mechanical fault" in path_text
        or "mechanical_fault" in path_text
        or "abnormal_vibration" in path_text
    ):
        return "mechanical_fault"

    if (
        "thermal fault" in path_text
        or "thermal_fault" in path_text
        or "abnormal_temp" in path_text
    ):
        return "thermal_fault"

    if (
        "/normal/" in path_text
        or stem.startswith("normal_")
        or stem == "normal"
    ):
        return "normal"

    return None


In [ ]:
# ==========================================================
# CELL 4 — discover and read CSV / Excel files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(
        os.path.join(base_path, "**", "*.xlsx"),
        recursive=True,
    )
    files += glob.glob(
        os.path.join(base_path, "**", "*.csv"),
        recursive=True,
    )
    return sorted(set(files))


def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    missing_columns = [
        column
        for column in FEATURE_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in {path.name}: {missing_columns}"
        )

    clean = df[FEATURE_COLUMNS].apply(
        pd.to_numeric,
        errors="coerce",
    )

    clean = clean / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan)
    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(
            f"{path.name} has only {len(clean)} valid rows."
        )

    return clean

In [ ]:
# ==========================================================
# CELL 5 — load safe and fault sessions
# ==========================================================
files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: {BASE_PATH}"
    )

sessions = []

for file_path in files:
    state = state_for(file_path)

    if state is None:
        print(
            "SKIP unrecognized:",
            os.path.relpath(file_path, BASE_PATH),
        )
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(file_path, BASE_PATH),
            "|",
            error,
        )
        continue

    sessions.append({
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "state": state,
        "binary_label": 0 if state in SAFE_STATES else 1,
        "rows": len(df),
    })

if not sessions:
    raise RuntimeError("No sessions were loaded.")

state_counts = pd.Series([
    session["state"]
    for session in sessions
]).value_counts().reindex(
    SAFE_STATES + FAULT_STATES,
    fill_value=0,
)

print("\nFILES PER STATE")
print(state_counts)

missing_states = [
    state
    for state in SAFE_STATES + FAULT_STATES
    if state_counts[state] == 0
]

if missing_states:
    raise RuntimeError(
        f"No files found for states: {missing_states}"
    )

safe_sessions = [
    session
    for session in sessions
    if session["binary_label"] == 0
]

fault_sessions = [
    session
    for session in sessions
    if session["binary_label"] == 1
]


In [ ]:
# ==========================================================
# CELL 6 — scaling, window creation, and labeled datasets
# ==========================================================
def scale_array(X_raw, scaler):
    X_scaled = scaler.transform(X_raw).astype(np.float32)

    if CLIP_SCALED_INPUT:
        X_scaled = np.clip(X_scaled, 0.0, 1.0)

    return X_scaled


def make_windows_from_array(X_raw, scaler):
    X_scaled = scale_array(X_raw, scaler)

    starts = np.arange(
        0,
        len(X_scaled) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_scaled[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return windows, starts


def build_windows_from_sessions(selected_sessions, scaler):
    all_windows = []

    for session in selected_sessions:
        windows, _ = make_windows_from_array(
            session["X"],
            scaler,
        )

        if windows is not None:
            all_windows.append(windows)

    if not all_windows:
        raise RuntimeError("No windows were created.")

    return np.concatenate(all_windows, axis=0)


def build_labeled_windows_from_sessions(selected_sessions, scaler):
    all_windows = []
    all_labels = []
    all_files = []
    all_starts = []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(
            session["X"],
            scaler,
        )

        if windows is None:
            continue

        all_windows.append(windows)
        all_labels.extend([session["binary_label"]] * len(windows))
        all_files.extend([session["file"]] * len(windows))
        all_starts.extend(starts.tolist())

    if not all_windows:
        raise RuntimeError("No labeled windows were created.")

    return (
        np.concatenate(all_windows, axis=0),
        np.asarray(all_labels, dtype=np.float32),
        np.asarray(all_files, dtype=object),
        np.asarray(all_starts, dtype=np.int64),
    )

In [ ]:
# ==========================================================
# CELL 7 — callbacks
# ==========================================================
def make_callbacks():
    return [
        EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=15,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

In [ ]:
# ==========================================================
# CELL 8 — build Binary Classifier
# ==========================================================
def build_classifier():
    inputs = Input(
        shape=(WINDOW_LEN, N_FEATURES),
        name="float_input",
    )

    x = Flatten(name="flatten_window")(inputs)

    x = Dense(
        units=32,
        activation="relu",
        name="dense_1",
    )(x)
    x = Dropout(0.20, name="dropout_1")(x)

    x = Dense(
        units=16,
        activation="relu",
        name="dense_2",
    )(x)
    x = Dropout(0.10, name="dropout_2")(x)

    outputs = Dense(
        units=1,
        activation="sigmoid",
        name="binary_output",
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="binary_classifier",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3,
        ),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
        ],
    )

    return model

In [ ]:
# ==========================================================
# CELL 9 — class weights for imbalanced safe/fault data
# ==========================================================
from sklearn.utils.class_weight import compute_class_weight

def calculate_binary_class_weights(y_train):
    present_classes = np.unique(y_train.astype(np.int64))

    weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_train.astype(np.int64),
    )

    return {
        int(class_id): float(weight)
        for class_id, weight in zip(present_classes, weights)
    }

In [ ]:
# ==========================================================
# CELL 10 — window and file predictions
# ==========================================================
def aggregate_predictions(
    selected_sessions,
    model,
    scaler,
    decision_threshold=0.5,
):
    file_rows = []
    window_rows = []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(
            session["X"],
            scaler,
        )

        if windows is None:
            continue

        probabilities = model.predict(windows, verbose=0).reshape(-1)
        flags = (probabilities >= decision_threshold).astype(np.int64)

        anomaly_fraction = float(np.mean(flags))
        file_prediction = int(
            anomaly_fraction >= FILE_ANOMALY_FRACTION_THRESHOLD
        )

        file_rows.append({
            "file": session["file"],
            "state": session["state"],
            "true_label": session["binary_label"],
            "pred_label": file_prediction,
            "windows": len(windows),
            "anomaly_windows": int(np.sum(flags)),
            "anomaly_fraction": anomaly_fraction,
            "mean_probability": float(np.mean(probabilities)),
            "maximum_probability": float(np.max(probabilities)),
        })

        for window_id in range(len(windows)):
            window_rows.append({
                "file": session["file"],
                "state": session["state"],
                "window": window_id,
                "start_row": int(starts[window_id]),
                "true_label": session["binary_label"],
                "pred_label": int(flags[window_id]),
                "probability": float(probabilities[window_id]),
            })

    return (
        pd.DataFrame(file_rows),
        pd.DataFrame(window_rows),
    )

In [ ]:
# ==========================================================
# CELL 11 — five-fold file-wise cross-validation
# ==========================================================
binary_counts = pd.Series([
    session["binary_label"]
    for session in sessions
]).value_counts()

if binary_counts.min() < N_SPLITS:
    raise RuntimeError(
        f"Need at least {N_SPLITS} safe files and "
        f"{N_SPLITS} fault files. Current: "
        f"{binary_counts.to_dict()}"
    )

cv_table = pd.DataFrame({
    "label": [
        session["binary_label"]
        for session in sessions
    ],
    "group": [
        session["file"]
        for session in sessions
    ],
})

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
cv_file_results = []
cv_window_results = []

for fold, (train_index, test_index) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    print("\n" + "=" * 60)
    print("FOLD", fold)
    print("=" * 60)

    train_sessions = [sessions[index] for index in train_index]
    test_sessions = [sessions[index] for index in test_index]

    rng = np.random.default_rng(SEED + fold)
    fitting_sessions = []
    validation_sessions = []

    for binary_value in [0, 1]:
        label_sessions = [
            session
            for session in train_sessions
            if session["binary_label"] == binary_value
        ]

        if len(label_sessions) < 2:
            raise RuntimeError(
                f"Fold {fold}: binary label {binary_value} requires "
                "at least two training files."
            )

        shuffled_ids = rng.permutation(len(label_sessions))
        number_validation = max(1, round(0.20 * len(label_sessions)))
        number_validation = min(number_validation, len(label_sessions) - 1)
        validation_ids = set(shuffled_ids[:number_validation])

        for local_index, session in enumerate(label_sessions):
            if local_index in validation_ids:
                validation_sessions.append(session)
            else:
                fitting_sessions.append(session)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(
        np.concatenate([
            session["X"]
            for session in fitting_sessions
        ])
    )

    X_train, y_train, _, _ = build_labeled_windows_from_sessions(
        fitting_sessions,
        scaler,
    )

    X_validation, y_validation, _, _ = build_labeled_windows_from_sessions(
        validation_sessions,
        scaler,
    )

    class_weight = calculate_binary_class_weights(y_train)

    tf.keras.backend.clear_session()
    model = build_classifier()

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=make_callbacks(),
        class_weight=class_weight,
        shuffle=True,
        verbose=1,
    )

    fold_file_df, fold_window_df = aggregate_predictions(
        test_sessions,
        model,
        scaler,
    )

    fold_file_df.insert(0, "fold", fold)
    fold_window_df.insert(0, "fold", fold)

    cv_file_results.append(fold_file_df)
    cv_window_results.append(fold_window_df)

    y_true = fold_window_df["true_label"].to_numpy()
    y_pred = fold_window_df["pred_label"].to_numpy()
    y_score = fold_window_df["probability"].to_numpy()

    fold_metrics = {
        "fold": fold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": (
            roc_auc_score(y_true, y_score)
            if len(np.unique(y_true)) == 2
            else np.nan
        ),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
    }

    cv_metrics.append(fold_metrics)
    print(pd.Series(fold_metrics))

In [ ]:
# ==========================================================
# CELL 12 — combined cross-validation report
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_file_results_df = pd.concat(cv_file_results, ignore_index=True)
cv_window_results_df = pd.concat(cv_window_results, ignore_index=True)

cv_metrics_df.to_csv(os.path.join(CV_DIR, "cv_metrics.csv"), index=False)
cv_file_results_df.to_csv(os.path.join(CV_DIR, "cv_file_predictions.csv"), index=False)
cv_window_results_df.to_csv(os.path.join(CV_DIR, "cv_window_predictions.csv"), index=False)

window_true = cv_window_results_df["true_label"].to_numpy()
window_pred = cv_window_results_df["pred_label"].to_numpy()
window_score = cv_window_results_df["probability"].to_numpy()

file_true = cv_file_results_df["true_label"].to_numpy()
file_pred = cv_file_results_df["pred_label"].to_numpy()

overall_metrics = {
    "window_accuracy": accuracy_score(window_true, window_pred),
    "window_balanced_accuracy": balanced_accuracy_score(window_true, window_pred),
    "window_precision": precision_score(window_true, window_pred, zero_division=0),
    "window_recall": recall_score(window_true, window_pred, zero_division=0),
    "window_f1": f1_score(window_true, window_pred, zero_division=0),
    "window_roc_auc": roc_auc_score(window_true, window_score),
    "file_accuracy": accuracy_score(file_true, file_pred),
    "file_balanced_accuracy": balanced_accuracy_score(file_true, file_pred),
    "file_f1": f1_score(file_true, file_pred, zero_division=0),
}

print("\nOVERALL CROSS-VALIDATION RESULTS")
print(pd.Series(overall_metrics))

print("\nWINDOW REPORT")
print(classification_report(
    window_true, window_pred, target_names=["safe", "fault"], zero_division=0,
))

print("\nFILE REPORT")
print(classification_report(
    file_true, file_pred, target_names=["safe", "fault"], zero_division=0,
))

pd.DataFrame([overall_metrics]).to_csv(
    os.path.join(CV_DIR, "overall_metrics.csv"), index=False,
)

ConfusionMatrixDisplay(
    confusion_matrix(window_true, window_pred, labels=[0, 1]),
    display_labels=["safe", "fault"],
).plot(values_format="d")

plt.title(f"{MODEL_DISPLAY_NAME} Cross-Validation")
plt.tight_layout()
plt.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)
plt.show()

In [ ]:
# ==========================================================
# CELL 13 — final train/validation split (stratified by binary label)
# ==========================================================
rng = np.random.default_rng(SEED)
final_train_sessions = []
final_validation_sessions = []

for binary_value in [0, 1]:
    label_sessions = [
        session
        for session in sessions
        if session["binary_label"] == binary_value
    ]

    if len(label_sessions) < 2:
        raise RuntimeError(
            f"Binary label {binary_value} needs at least two files."
        )

    shuffled_ids = rng.permutation(len(label_sessions))
    number_validation = max(1, round(0.20 * len(label_sessions)))
    number_validation = min(number_validation, len(label_sessions) - 1)
    validation_ids = set(shuffled_ids[:number_validation])

    for local_index, session in enumerate(label_sessions):
        if local_index in validation_ids:
            final_validation_sessions.append(session)
        else:
            final_train_sessions.append(session)

final_scaler = MinMaxScaler(feature_range=(0, 1))
final_scaler.fit(
    np.concatenate([
        session["X"]
        for session in final_train_sessions
    ])
)

X_final_train, y_final_train, _, _ = build_labeled_windows_from_sessions(
    final_train_sessions,
    final_scaler,
)

X_final_validation, y_final_validation, validation_files, validation_starts = (
    build_labeled_windows_from_sessions(
        final_validation_sessions,
        final_scaler,
    )
)

final_class_weight = calculate_binary_class_weights(y_final_train)

print("Final training windows:", X_final_train.shape)
print("Final validation windows:", X_final_validation.shape)
print("Class weights:", final_class_weight)

In [ ]:
# ==========================================================
# CELL 14 — train final deployment model
# ==========================================================
tf.keras.backend.clear_session()
final_model = build_classifier()

final_history = final_model.fit(
    X_final_train,
    y_final_train,
    validation_data=(X_final_validation, y_final_validation),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    class_weight=final_class_weight,
    shuffle=True,
    verbose=1,
)

validation_probabilities = final_model.predict(X_final_validation, verbose=0).reshape(-1)
validation_predictions = (validation_probabilities >= 0.5).astype(np.int64)

print("\nFINAL VALIDATION REPORT")
print(classification_report(
    y_final_validation, validation_predictions,
    target_names=["safe", "fault"], zero_division=0,
))

print("Balanced accuracy:", balanced_accuracy_score(y_final_validation, validation_predictions))
print("F1:", f1_score(y_final_validation, validation_predictions, zero_division=0))
print("ROC AUC:", roc_auc_score(y_final_validation, validation_probabilities))

plt.figure(figsize=(9, 4))
plt.plot(final_history.history["loss"], label="training loss")
plt.plot(final_history.history["val_loss"], label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary crossentropy loss")
plt.title(f"{MODEL_DISPLAY_NAME} Training")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FINAL_DIR, "training_loss.png"), dpi=150)
plt.show()

In [ ]:
# ==========================================================
# CELL 15 — save model, scaler, labels, metadata
# ==========================================================
KERAS_PATH = os.path.join(FINAL_DIR, f"{MODEL_FILE_PREFIX}.keras")
SCALER_PATH = os.path.join(FINAL_DIR, f"{MODEL_FILE_PREFIX}_scaler.pkl")
LABELS_PATH = os.path.join(FINAL_DIR, f"{MODEL_FILE_PREFIX}_labels.txt")
METADATA_PATH = os.path.join(FINAL_DIR, f"{MODEL_FILE_PREFIX}_metadata.csv")

final_model.save(KERAS_PATH)
joblib.dump(final_scaler, SCALER_PATH)

with open(LABELS_PATH, "w", encoding="utf-8") as file:
    file.write("safe\nfault")

pd.DataFrame({
    "feature_id": np.arange(N_FEATURES),
    "feature_name": FEATURE_COLUMNS,
    "scaler_min": final_scaler.data_min_,
    "scaler_max": final_scaler.data_max_,
}).to_csv(METADATA_PATH, index=False)

print("Saved:", KERAS_PATH)
print("Saved:", SCALER_PATH)
print("Saved:", LABELS_PATH)
print("Saved:", METADATA_PATH)

In [ ]:
# ==========================================================
# CELL 16 — export Float32 and dynamic-range TFLite
# ==========================================================
FLOAT32_TFLITE_PATH = os.path.join(FINAL_DIR, f"{MODEL_FILE_PREFIX}_float32.tflite")
DYNAMIC_TFLITE_PATH = os.path.join(FINAL_DIR, f"{MODEL_FILE_PREFIX}_dynamic.tflite")

converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
float32_tflite = converter.convert()

with open(FLOAT32_TFLITE_PATH, "wb") as file:
    file.write(float32_tflite)

converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
dynamic_tflite = converter.convert()

with open(DYNAMIC_TFLITE_PATH, "wb") as file:
    file.write(dynamic_tflite)

print("Saved:", FLOAT32_TFLITE_PATH)
print("Saved:", DYNAMIC_TFLITE_PATH)

In [ ]:
# ==========================================================
# CELL 17 — export ONNX
# ==========================================================
import tf2onnx

ONNX_PATH = os.path.join(FINAL_DIR, f"{MODEL_FILE_PREFIX}.onnx")

input_signature = (
    tf.TensorSpec((None, WINDOW_LEN, N_FEATURES), tf.float32, name="float_input"),
)

tf2onnx.convert.from_keras(
    final_model,
    input_signature=input_signature,
    opset=13,
    output_path=ONNX_PATH,
)

print("Saved:", ONNX_PATH)

In [ ]:
# ==========================================================
# CELL 18 — export complete direct C header
# ==========================================================
HEADER_PATH = os.path.join(FINAL_DIR, f"{MODEL_FILE_PREFIX}_model.h")


def write_float_array(file, name, values):
    values = np.asarray(values, dtype=np.float32).reshape(-1)

    file.write(f"static const float {name}[{len(values)}] = {{\n")

    for start in range(0, len(values), 8):
        line = ", ".join(
            f"{float(value):.9g}f"
            for value in values[start:start + 8]
        )
        file.write(f"    {line},\n")

    file.write("};\n\n")


with open(HEADER_PATH, "w", encoding="utf-8") as file:
    file.write("#ifndef MOTOR_BINARY_CLASSIFIER_MODEL_H\n")
    file.write("#define MOTOR_BINARY_CLASSIFIER_MODEL_H\n\n")
    file.write("#include <math.h>\n\n")

    file.write(f"#define BC_WL {WINDOW_LEN}\n")
    file.write(f"#define BC_NF {N_FEATURES}\n")
    file.write(f"#define BC_INPUT {INPUT_SIZE}\n")
    file.write("#define BC_H1 32\n")
    file.write("#define BC_H2 16\n")
    file.write(f"#define BC_DIV {SCALE_DIVISOR:.9g}f\n\n")

    write_float_array(file, "BC_MIN", final_scaler.data_min_)
    write_float_array(file, "BC_MAX", final_scaler.data_max_)

    W1, B1 = final_model.get_layer("dense_1").get_weights()
    W2, B2 = final_model.get_layer("dense_2").get_weights()
    WO, BO = final_model.get_layer("binary_output").get_weights()

    write_float_array(file, "BC_W1", W1)
    write_float_array(file, "BC_B1", B1)
    write_float_array(file, "BC_W2", W2)
    write_float_array(file, "BC_B2", B2)
    write_float_array(file, "BC_WO", WO)
    write_float_array(file, "BC_BO", BO)

    c_source = r'''
static inline float bc_relu(float x)
{
    return x > 0.0f ? x : 0.0f;
}

static inline float bc_sigmoid(float x)
{
    return 1.0f / (1.0f + expf(-x));
}

static inline float binary_classifier_predict(
    const float raw[BC_WL][BC_NF])
{
    float input[BC_INPUT];
    float h1[BC_H1];
    float h2[BC_H2];
    float output;

    for (int t = 0; t < BC_WL; t++)
    {
        for (int f = 0; f < BC_NF; f++)
        {
            float physical = raw[t][f] / BC_DIV;
            float range = BC_MAX[f] - BC_MIN[f];

            input[t * BC_NF + f] =
                range > 0.0f
                ? (physical - BC_MIN[f]) / range
                : 0.0f;
        }
    }

    for (int j = 0; j < BC_H1; j++)
    {
        float sum = BC_B1[j];

        for (int i = 0; i < BC_INPUT; i++)
        {
            sum += input[i] * BC_W1[i * BC_H1 + j];
        }

        h1[j] = bc_relu(sum);
    }

    for (int j = 0; j < BC_H2; j++)
    {
        float sum = BC_B2[j];

        for (int i = 0; i < BC_H1; i++)
        {
            sum += h1[i] * BC_W2[i * BC_H2 + j];
        }

        h2[j] = bc_relu(sum);
    }

    {
        float sum = BC_BO[0];

        for (int i = 0; i < BC_H2; i++)
        {
            sum += h2[i] * BC_WO[i];
        }

        output = bc_sigmoid(sum);
    }

    return output;
}

/*
 * return 0 = safe
 * return 1 = fault
 */
static inline int binary_classifier_decision(float probability)
{
    return probability >= 0.5f;
}

#endif
'''

    file.write(c_source)

print("Saved:", HEADER_PATH)
print("Header size KB:", round(os.path.getsize(HEADER_PATH) / 1024, 2))

In [ ]:
# ==========================================================
# CELL 18.1 — export parameter-only header (for TFLite runtime)
# ==========================================================
PARAMS_HEADER_PATH = os.path.join(
    FINAL_DIR,
    f"{MODEL_FILE_PREFIX}_params.h",
)

with open(PARAMS_HEADER_PATH, "w", encoding="utf-8") as file:
    file.write("#ifndef BINARY_CLASSIFIER_PARAMS_H\n")
    file.write("#define BINARY_CLASSIFIER_PARAMS_H\n\n")
    file.write("#include <stdint.h>\n\n")

    file.write(
        """/*
 * Parameters for:
 * binary_autoencoder_float32.tflite  (now a binary classifier)
 *
 * TFLite input:
 *     1 x 30 x 14 FLOAT32
 *
 * TFLite output:
 *     1 x 1 FLOAT32 sigmoid probability
 *     >= 0.5 -> fault, < 0.5 -> safe
 *
 * The TFLite model does not contain an internal scaler.
 * Apply MinMax scaling before inference.
 *
 * The STM32 features are already physical floating-point
 * values. Do not divide the live STM32 values by 1,000,000.
 */

"""
    )

    file.write(f"#define BC_WINDOW_LEN {WINDOW_LEN}U\n")
    file.write(f"#define BC_N_FEATURES {N_FEATURES}U\n")
    file.write(f"#define BC_INPUT_SIZE {INPUT_SIZE}U\n\n")

    file.write(
        "#define BC_FILE_ANOMALY_FRACTION_THRESHOLD "
        f"{FILE_ANOMALY_FRACTION_THRESHOLD:.9g}f\n\n"
    )

    file.write("#define BC_SAFE_STATE  0U\n")
    file.write("#define BC_FAULT_STATE 1U\n\n")

    file.write(
        "/* Exact input-feature order used during training. */\n"
    )
    file.write(
        f"static const char *const BC_FEATURE_NAMES[BC_N_FEATURES] = {{\n"
    )
    for feature_name in FEATURE_COLUMNS:
        file.write(f'    "{feature_name}",\n')
    file.write("};\n\n")

    file.write(
        f"static const float BC_MIN[BC_N_FEATURES] = {{\n    "
    )
    file.write(
        ", ".join(f"{v:.9g}f" for v in final_scaler.data_min_)
    )
    file.write("\n};\n\n")

    file.write(
        f"static const float BC_MAX[BC_N_FEATURES] = {{\n    "
    )
    file.write(
        ", ".join(f"{v:.9g}f" for v in final_scaler.data_max_)
    )
    file.write("\n};\n\n")

    file.write(
        r"""/*
 * MinMaxScaler transformation used during training.
 */
static inline float bc_minmax_scale(
    float physical_value,
    uint32_t feature_index)
{
    const float minimum = BC_MIN[feature_index];
    const float maximum = BC_MAX[feature_index];
    const float range = maximum - minimum;

    if (range <= 1.0e-12f)
    {
        return 0.0f;
    }

    return (physical_value - minimum) / range;
}


/*
 * Window-level decision from the TFLite sigmoid output.
 *
 * return 0 = safe
 * return 1 = fault
 */
static inline uint32_t bc_is_fault(float probability)
{
    return probability >= 0.5f
        ? BC_FAULT_STATE
        : BC_SAFE_STATE;
}

#endif /* BINARY_CLASSIFIER_PARAMS_H */
"""
    )

print("Saved:", PARAMS_HEADER_PATH)

In [ ]:
# ==========================================================
# CELL 19 — manual classification test
# ==========================================================
def manual_binary_prediction(test_file):
    model = tf.keras.models.load_model(KERAS_PATH)
    scaler = joblib.load(SCALER_PATH)

    test_df = read_one(test_file)
    X_test, starts = make_windows_from_array(
        test_df.to_numpy(dtype=np.float32),
        scaler,
    )

    probabilities = model.predict(X_test, verbose=0).reshape(-1)
    flags = (probabilities >= 0.5).astype(np.int64)
    anomaly_fraction = float(np.mean(flags))

    file_decision = (
        "FAULT"
        if anomaly_fraction >= FILE_ANOMALY_FRACTION_THRESHOLD
        else "SAFE"
    )

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (starts + WINDOW_LEN / 2.0) * ROW_PERIOD_SEC,
        "probability": probabilities,
        "prediction": flags,
        "decision": np.where(flags == 1, "fault", "safe"),
    })

    stem = Path(test_file).stem
    output_csv = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}.csv")
    result.to_csv(output_csv, index=False)

    print("\n" + "=" * 60)
    print(f"MANUAL {MODEL_DISPLAY_NAME} TEST")
    print("=" * 60)
    print("File:", test_file)
    print("Total windows:", len(result))
    print("Fault windows:", int(np.sum(flags)))
    print("Fault fraction:", anomaly_fraction)
    print("File decision:", file_decision)
    print("Saved CSV:", output_csv)

    plt.figure(figsize=(12, 4))
    plt.plot(result["time_sec"], result["probability"])
    plt.axhline(0.5, linestyle="--", label="decision threshold")
    plt.xlabel("Time [s]")
    plt.ylabel("Fault probability")
    plt.title(f"{MODEL_DISPLAY_NAME} — {stem}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_probability.png"),
        dpi=150,
    )
    plt.show()

    return result




In [ ]:
# ==========================================================
# CELL 20 — select manual test file
# ==========================================================
if __name__ == "__main__":
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    TEST_FILE = filedialog.askopenfilename(
        title="Select motor CSV or Excel file",
        filetypes=[
            ("Excel files", "*.xlsx"),
            ("CSV files", "*.csv"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if TEST_FILE:
        manual_result = manual_binary_prediction(TEST_FILE)
    else:
        print("No file selected.")